# ETL BANCI — edición autónoma

Replica el flujo de [2_Procesamiento_ETL_BANCI.ipynb](2_Procesamiento_ETL_BANCI.ipynb) con sus dependencias funcionales trasladadas a este mismo archivo: lectura optimizada, alias, validación de entrega contra el Parquet previo, reparación de nombres, CURP, fechas, llaves, campos derivados y exportación. No importa `config.py` ni ningún módulo de `src/`.

**Ejecución:** corre las celdas de arriba hacia abajo. Las rutas y opciones están concentradas en la siguiente sección. La validación histórica estricta permanece activa; el Parquet existente se usa como referencia y se reemplaza solamente cuando el procesamiento termina correctamente.

## 1. Imports y configuración

In [ ]:
from __future__ import annotations

import json
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display


def encontrar_raiz(inicio: Path) -> Path:
    """Encuentra la raíz sin depender de la carpeta desde la que inicia Jupyter."""
    for candidato in (inicio.resolve(), *inicio.resolve().parents):
        if (candidato / "input" / "banci").is_dir() and (candidato / "output").is_dir():
            return candidato
    raise FileNotFoundError("No se encontró la raíz del proyecto con input/banci/ y output/.")


PROJECT_ROOT = encontrar_raiz(Path.cwd())
INPUT_BANCI = PROJECT_ROOT / "input" / "banci" / "SUBIR_R_202609_septiembre.xlsx"
SHEET_BANCI = "Victimas"
ETL_BANCI_OUTPUT_FILE = PROJECT_ROOT / "output" / "Resultado_2_Procesamiento_ETL_BANCI.parquet"
OUTPUT_FILE = ETL_BANCI_OUTPUT_FILE
EXPORTAR_RESULTADO = True
FORMATO_SALIDA = "parquet"
ESTADO = None
VALIDACION_ESTRICTA = True

BANCI_COLUMN_MAPPING = {
    "nombres": ["nombres", "nombre", "NOMBRES"],
    "primer_apellido": ["primer_apellido", "paterno", "apellido_paterno"],
    "segundo_apellido": ["segundo_apellido", "materno", "apellido_materno"],
    "curp": ["curp", "CURP", "curp_victima"],
    "sexo": ["sexo"],
    "fha_nac": ["fha_nac", "fecha_nacimiento", "fecha_nac"],
    "entidad": ["entidad", "entidad_federativa", "estado_reporta", "entidad_reporta", "ENTIDAD FEDERATIVA"],
    "id_mun_hchos": ["id_mun_hchos", "idmunicipio_hechos", "id_municipio_hechos"],
    "nom_mun_hchos": ["NOM_MUN_HCHOS", "nom_mun_hchos", "municipio_hechos", "nombre_municipio_hechos", "municipio"],
    "fha_de_hchos": ["fha_de_hchos", "fecha_hechos", "Fecha Hechos", "fecha de hechos"],
    "fha_de_ini": ["fha_de_ini", "fecha_inicio", "fecha inicio"],
    "localizado_o_no_localizado": ["localizado_o_no_localizado", "estatus", "Estatus", "estatus_localizado"],
    "fue_delito": ["fue_delito", "fue_delito_origen"],
    "delito_hom": ["delito_hom", "delito_homologado"],
    "llave": ["llave"],
    "folio_rnpdno": ["folio_rnpdno", "folio_rnpdno_ori", "folio_RNPDNO", "fub_ok"],
    "ntra_ci": ["NTRA_CI", "ntra_ci", "NtraCi"],
    "id_ci": ["ID_CI", "id_ci", "idci", "IdCi"],
}
BANCI_OUTPUT_COLUMNS = [
    "entidad_banci", "llave_banci", "id_ci_banci", "ntra_ci_banci", "fha_de_ini_banci",
    "fha_de_hchos_banci", "nom_mun_hchos_banci", "id_mun_hchos_banci", "delito_hom_banci",
    "sexo_banci", "fha_nac_banci", "folio_rnpdno_banci", "primer_apellido_banci",
    "segundo_apellido_banci", "nombres_banci", "curp_banci",
    "localizado_o_no_localizado_banci", "fue_delito_banci", "fub_ok_banci",
    "entidad_derivada_banci", "id_ci_derivada_banci", "ntra_ci_derivada_banci",
    "fha_de_hchos_derivada_banci", "nom_mun_hchos_derivada_banci",
    "localizado_o_no_localizado_derivada_banci", "fha_nac_derivada_banci",
    "nombres_derivada_banci", "primer_apellido_derivada_banci",
    "segundo_apellido_derivada_banci", "nombre_completo_derivada_banci",
    "curp_valida_derivada_banci", "id_carpeta_derivada_banci",
]
VALID_STATE_CODES = {
    "AS", "BC", "BS", "CC", "CL", "CM", "CS", "CH", "DF", "DG", "GT", "GR", "HG", "JC", "MC", "MN",
    "MS", "NT", "NL", "OC", "PL", "QT", "QR", "SP", "SL", "SR", "TC", "TS", "TL", "VZ", "YN", "ZS", "NE"
}
ENTIDADES = {
    1: "Aguascalientes", 2: "Baja California", 3: "Baja California Sur", 4: "Campeche", 5: "Coahuila", 6: "Colima",
    7: "Chiapas", 8: "Chihuahua", 9: "Ciudad de México", 10: "Durango", 11: "Guanajuato", 12: "Guerrero",
    13: "Hidalgo", 14: "Jalisco", 15: "México", 16: "Michoacán", 17: "Morelos", 18: "Nayarit", 19: "Nuevo León",
    20: "Oaxaca", 21: "Puebla", 22: "Querétaro", 23: "Quintana Roo", 24: "San Luis Potosí", 25: "Sinaloa",
    26: "Sonora", 27: "Tabasco", 28: "Tamaulipas", 29: "Tlaxcala", 30: "Veracruz", 31: "Yucatán", 32: "Zacatecas",
    33: "Nacido en el Extranjero",
}

print(f"Raíz: {PROJECT_ROOT}")
print(f"Entrada BANCI: {INPUT_BANCI} | hoja: {SHEET_BANCI}")
print(f"Salida ETL: {OUTPUT_FILE}")

## 2. Funciones auxiliares autónomas

Se trasladan las funciones de estandarización, limpieza, validación de CURP, reparación de nombres y validación/escritura del resumen histórico BANCI. Las reglas de validación estricta y umbrales corresponden al ETL original.

In [ ]:
def normalize_header(value):
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii").lower().strip()
    text = re.sub(r"[\s.\-]+", "_", text)
    return re.sub(r"[^a-z0-9_]", "", text)


def standardize_columns(df, mapping_dict, strict=False, verbose=True):
    out = df.copy()
    current = {normalize_header(c): c for c in out.columns}
    rename, missing = {}, []
    for target, aliases in mapping_dict.items():
        if target in out.columns:
            continue
        found = next((current[normalize_header(alias)] for alias in aliases if normalize_header(alias) in current), None)
        if found is None:
            missing.append(target)
        else:
            rename[found] = target
    if rename:
        if verbose:
            print(f"  🔄 Estandarizando columnas: Renombrando {len(rename)} columnas.")
            for source, target in rename.items(): print(f"     - '{source}' -> '{target}'")
        out = out.rename(columns=rename)
    if strict and missing:
        raise ValueError(f"No se encontraron columnas requeridas: {missing}")
    return out


def normalizar_texto(texto):
    replacements = {"Á":"A", "É":"E", "Í":"I", "Ó":"O", "Ú":"U", "Ü":"U"}
    if isinstance(texto, pd.Series):
        out = texto.fillna("").astype(str).str.upper()
        for a, b in replacements.items(): out = out.str.replace(a, b, regex=False)
        out = out.str.replace(r"\s+", " ", regex=True).str.strip()
        return out.mask(texto.isna(), None)
    if pd.isna(texto): return None
    out = str(texto).upper()
    for a, b in replacements.items(): out = out.replace(a, b)
    return re.sub(r"\s+", " ", out).strip()


def limpiar_curp(series):
    return series.astype(str).str.replace(r"\s+", "", regex=True).str.upper().replace(["NAN", "NONE", ""], np.nan)


def es_curp_valida_vectorizado(series):
    length_ok = series.str.len() == 18
    format_ok = series.str.contains(r"^[A-Z]{4}\d{6}[HM]", regex=True, na=False, flags=re.IGNORECASE)
    homoclave_ok = series.str.contains(r"[A-Z]{3}.{2}$", regex=True, na=False, flags=re.IGNORECASE)
    state_ok = series.str.slice(11, 13).str.upper().isin(VALID_STATE_CODES)
    return length_ok & format_ok & homoclave_ok & state_ok


def reparar_nombres_mal_formados(df):
    out = df.copy()
    names, last1, last2 = "nombres", "primer_apellido", "segundo_apellido"
    for col in (names, last1, last2):
        if col not in out.columns: out[col] = np.nan
    def split_row(row):
        nombre = str(row[names]) if pd.notna(row[names]) else ""
        ape1 = str(row[last1]) if pd.notna(row[last1]) else ""
        ape2 = str(row[last2]) if pd.notna(row[last2]) else ""
        if not (len(nombre) > 0 and (len(ape1) == 0 or len(ape2) == 0)):
            return row[names], row[last1], row[last2]
        limpio = re.sub(r"\s+", " ", nombre).strip()
        patron = r"\b(de\s+la|de\s+las|de\s+los|de|del|la|las|los|y)\s+"
        protegido = re.sub(patron, lambda m: m.group(0).replace(" ", "_"), limpio, flags=re.IGNORECASE)
        parts = [p for p in protegido.split(" ") if p]
        if len(parts) >= 3: values = (" ".join(parts[:-2]), parts[-2], parts[-1])
        elif len(parts) == 2: values = (parts[0], parts[1], None)
        else: values = (nombre, None, None)
        return tuple(v.replace("_", " ") if v else None for v in values)
    mask = out[names].notna() & (out[last1].isna() | out[last2].isna())
    if mask.sum():
        print(f"  > aplicando corrección de nombres a {int(mask.sum())} registros...")
        res = out.loc[mask].apply(split_row, axis=1, result_type="expand")
        out.loc[mask, [names, last1, last2]] = res.values
    return out


def corregir_anio_futuro(df, columna_fecha, fecha_limite):
    if columna_fecha not in df.columns: return df
    salida = df.copy()
    salida[columna_fecha] = pd.to_datetime(salida[columna_fecha], errors="coerce")
    limite = pd.to_datetime(fecha_limite)
    def corregir(fecha):
        if pd.isnull(fecha): return fecha
        while fecha > limite:
            try: fecha = fecha.replace(year=fecha.year - 1)
            except ValueError: fecha = fecha.replace(month=2, day=28, year=fecha.year - 1)
        return fecha
    salida[columna_fecha] = salida[columna_fecha].apply(corregir)
    return salida


def _norm_entity(value):
    text = unicodedata.normalize("NFD", str(value)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", text).upper().strip()

ENTIDADES = {1:"Aguascalientes",2:"Baja California",3:"Baja California Sur",4:"Campeche",5:"Coahuila",6:"Colima",7:"Chiapas",8:"Chihuahua",9:"Ciudad de México",10:"Durango",11:"Guanajuato",12:"Guerrero",13:"Hidalgo",14:"Jalisco",15:"México",16:"Michoacán",17:"Morelos",18:"Nayarit",19:"Nuevo León",20:"Oaxaca",21:"Puebla",22:"Querétaro",23:"Quintana Roo",24:"San Luis Potosí",25:"Sinaloa",26:"Sonora",27:"Tabasco",28:"Tamaulipas",29:"Tlaxcala",30:"Veracruz",31:"Yucatán",32:"Zacatecas",33:"Nacido en el Extranjero"}
_MAPA_ENTIDAD = {_norm_entity(v): k for k, v in ENTIDADES.items()}
_MAPA_ENTIDAD.update({"ESTADO DE MEXICO":15,"MEXICO":15,"EDOMEX":15,"DISTRITO FEDERAL":9,"CDMX":9,"COAHUILA DE ZARAGOZA":5,"MICHOACAN DE OCAMPO":16,"VERACRUZ DE IGNACIO DE LA LLAVE":30})
_PATRON_ESTATUS = re.compile(r"^(NO )?LOCALIZAD[AO]S?$")
_SEXOS_CONOCIDOS = {"HOMBRE", "MUJER", "SE DESCONOCE", "NO ESPECIFICA", "NO ESPECIFICADO"}
UMBRAL_LLAVE_VACIA, UMBRAL_CRECIMIENTO, UMBRAL_BAJA_ENTIDAD, MINIMO_BAJA_ENTIDAD = 0.05, 0.50, 0.05, 10
CLAVE_METADATA_RESUMEN = b"banci_resumen"


def _fechas_validacion(s):
    f = pd.to_datetime(s, errors="coerce"); n = pd.to_numeric(s, errors="coerce")
    excel = pd.to_datetime(n, unit="D", origin="1899-12-30", errors="coerce")
    return f.where(f.notna(), excel.where(n.between(1, 100_000)))

def _texto_validacion(s): return s.map(lambda x: "" if pd.isna(x) else _norm_entity(x))
def _inicio_valido(df):
    ini = _fechas_validacion(df["fha_de_ini"]); futuras = ini > pd.Timestamp(datetime.now().date())
    return ini.where(~futuras), ini[futuras]

def resumir_entrega(df, archivo):
    ini, _ = _inicio_valido(df); cve = _texto_validacion(df["entidad"]).map(_MAPA_ENTIDAD)
    return {"archivo":Path(archivo).name,"procesado":datetime.now().isoformat(timespec="seconds"),"filas":int(len(df)),
            "fecha_corte":None if ini.isna().all() else ini.max().strftime("%Y-%m-%d"),
            "fecha_inicio_min":None if ini.isna().all() else ini.min().strftime("%Y-%m-%d"),
            "por_entidad":{str(int(k)):int(v) for k,v in cve.value_counts().items()}}

def leer_referencia(ruta):
    ruta=Path(ruta)
    if not ruta.exists():return None
    metadata=pq.read_schema(ruta).metadata or {}
    if CLAVE_METADATA_RESUMEN in metadata:return json.loads(metadata[CLAVE_METADATA_RESUMEN])
    df=pd.read_parquet(ruta,columns=["fha_de_ini_banci","entidad_banci"]).rename(columns={"fha_de_ini_banci":"fha_de_ini","entidad_banci":"entidad"})
    res=resumir_entrega(df,f"(reconstruido de {ruta.name})");res["procesado"]=None;return res

def validar_entrega_banci(df,archivo,referencia=None,estricta=True,verbose=True):
    res=resumir_entrega(df,archivo); errores=[];avisos=[]
    if not res["filas"]:errores.append("El archivo no trae registros.")
    ini,futuras=_inicio_valido(df);ilegibles=int((ini.isna()&~ini.index.isin(futuras.index)).sum())
    if len(futuras):avisos.append(f"{len(futuras):,} fechas de inicio posteriores a hoy; no cuentan para el corte.")
    if res["fecha_corte"] is None:errores.append("Ninguna fecha de inicio legible.")
    elif ilegibles:avisos.append(f"{ilegibles:,} registros sin fecha de inicio legible.")
    est=_texto_validacion(df["localizado_o_no_localizado"]);raros=est[~est.str.match(_PATRON_ESTATUS)].replace("","(vacío)")
    if len(raros):errores.append(f"{len(raros):,} registros con estatus no reconocido: {raros.value_counts().head(5).to_dict()}.")
    ent=_texto_validacion(df["entidad"]);sin=ent[ent.map(_MAPA_ENTIDAD).isna()].replace("","(vacío)")
    if len(sin):errores.append(f"{len(sin):,} registros con entidad sin clave INEGI: {sin.value_counts().head(5).to_dict()}.")
    sexo=_texto_validacion(df["sexo"]);nuevos=sexo[(sexo!="")&~sexo.isin(_SEXOS_CONOCIDOS)]
    if len(nuevos):avisos.append(f"Categorías de sexo nuevas: {nuevos.value_counts().head(5).to_dict()}.")
    vacias=df["llave"].isna()|(df["llave"].astype(str).str.strip()=="")
    if vacias.mean()>UMBRAL_LLAVE_VACIA:errores.append(f"{int(vacias.sum()):,} registros ({vacias.mean():.0%}) sin LLAVE.")
    elif vacias.any():avisos.append(f"{int(vacias.sum()):,} registros sin LLAVE.")
    if res["fecha_corte"]:
        n=int((_fechas_validacion(df["fha_de_hchos"])>pd.Timestamp(res["fecha_corte"])).sum())
        if n:avisos.append(f"{n:,} fechas de hechos posteriores al corte; se corregirán restando años.")
    comparacion="sin entrega anterior: esta queda como referencia"
    if referencia and referencia.get("fecha_corte") and res["fecha_corte"]:
        ant,nue=referencia["fecha_corte"],res["fecha_corte"];n_ant,n_nue=referencia["filas"],res["filas"]
        if nue<ant:errores.append(f"La entrega es de corte ANTERIOR ({nue}) a {ant} ({referencia['archivo']}).");comparacion="corte anterior"
        elif nue==ant:
            comparacion="mismo corte"
            if n_nue!=n_ant:avisos.append(f"Mismo corte pero cambió el total: {n_nue-n_ant:+,}.")
        else:
            comparacion="corte nuevo"
            if n_nue<=n_ant:errores.append(f"El corte nuevo debe crecer: {n_ant:,} ({ant}) → {n_nue:,} ({nue}).")
            elif n_nue/n_ant-1>UMBRAL_CRECIMIENTO:avisos.append(f"Creció {n_nue/n_ant-1:.0%} ({n_ant:,} → {n_nue:,}).")
            meses=(pd.Timestamp(nue).to_period("M")-pd.Timestamp(ant).to_period("M")).n
            if meses>1:avisos.append(f"Entre cortes hay {meses} meses; quizá falta una entrega.")
            if res["fecha_inicio_min"]!=referencia.get("fecha_inicio_min"):avisos.append(f"Cambió fecha de inicio mínima ({referencia.get('fecha_inicio_min')} → {res['fecha_inicio_min']}).")
            for cve,n in sorted(referencia.get("por_entidad",{}).items(),key=lambda kv:int(kv[0])):
                m=res["por_entidad"].get(cve,0);nombre=ENTIDADES.get(int(cve),cve)
                if m==0 and n>=MINIMO_BAJA_ENTIDAD:errores.append(f"La entidad {nombre} desaparece ({n:,} → 0).")
                elif n-m>=MINIMO_BAJA_ENTIDAD and (n-m)/n>UMBRAL_BAJA_ENTIDAD:avisos.append(f"La entidad {nombre} baja {(n-m)/n:.0%} ({n:,} → {m:,}).")
    if verbose:
        print(f"\n🔎 Validación BANCI: {res['archivo']} · {res['filas']:,} filas · corte {res['fecha_corte']} · inicio {res['fecha_inicio_min']} · {len(res['por_entidad'])} entidades")
        print(f"   {comparacion}")
        for a in avisos:print(f"   ⚠️ {a}")
        for e in errores:print(f"   ❌ {e}")
        if not errores and not avisos:print("   ✅ Sin observaciones.")
    if errores:
        msg="❌ La entrega de BANCI no pasó validaciones:\n"+"\n".join(f"   - {e}" for e in errores)
        if estricta:raise InsumoInvalido(msg)
        print(msg.replace("❌","⚠️",1)+"\n   (modo no estricto: se continúa)")
    return res

def escribir_parquet_con_resumen(df,ruta,resumen):
    tabla=pa.Table.from_pandas(df,preserve_index=False);meta=dict(tabla.schema.metadata or {})
    meta[CLAVE_METADATA_RESUMEN]=json.dumps(resumen,ensure_ascii=False).encode("utf-8")
    ruta=Path(ruta);ruta.parent.mkdir(parents=True,exist_ok=True);pq.write_table(tabla.replace_schema_metadata(meta),ruta)

def validar_columnas(df,columnas,fuente):
    faltan=[c for c in columnas if c not in df.columns]
    if faltan:raise ValueError(f"❌ ERROR CRÍTICO: '{fuente}' no contiene columnas requeridas: {faltan}")
    print(f"✅ Validación de columnas completada para '{fuente}'.")


def clean_col_text(col_name):return str(col_name).lower().strip().replace(" ","_").replace("(","").replace(")","")
def clean_col_names(df):
    out=df.copy();out.columns=[clean_col_text(c) for c in out.columns];return out

print("Helpers autónomos listos; el cuaderno no importa módulos del proyecto.")


In [ ]:
class InsumoInvalido(Exception):
    """Error de validación estricta del insumo BANCI, definido en esta notebook."""


## 3. Carga, estandarización y validación de la entrega

In [ ]:
# Equivalente al filtro usecols y al fallback del notebook de producción.
def clean_col_text(col_name):
    return str(col_name).lower().strip().replace(" ", "_").replace("(", "").replace(")", "")


def clean_col_names(df):
    df = df.copy()
    df.columns = [clean_col_text(c) for c in df.columns]
    return df


COLS_BANCI_TARGET = set(BANCI_COLUMN_MAPPING.keys())
valid_banci_inputs = set()
for target in COLS_BANCI_TARGET:
    valid_banci_inputs.add(target)
    for map_key, map_variations in BANCI_COLUMN_MAPPING.items():
        if target in map_variations:
            valid_banci_inputs.update(clean_col_text(v) for v in map_variations)


def filter_banci_cols(col_name):
    if not isinstance(col_name, str):
        return False
    return clean_col_text(col_name) in valid_banci_inputs


if not INPUT_BANCI.is_file():
    raise FileNotFoundError(f"No se encontró el Excel BANCI: {INPUT_BANCI}")

banci_sheet_desc = f", hoja: '{SHEET_BANCI}'" if SHEET_BANCI is not None else " (hoja predeterminada)"
print(f"Intentando cargar BANCI desde: {INPUT_BANCI}{banci_sheet_desc}")
print("  ⚡ Aplicando filtro de columnas 'usecols' en BANCI...")
try:
    banci_sheet_name = SHEET_BANCI if SHEET_BANCI is not None else 0
    df_banci = pd.read_excel(INPUT_BANCI, sheet_name=banci_sheet_name, usecols=filter_banci_cols)
    print(f"  ✅ Carga optimizada exitosa: {len(df_banci):,} registros.")
except Exception as error:
    print(f"  ⚠️ Falló usecols en BANCI: {error}. Cargando todo el archivo (lento)...")
    banci_sheet_name = SHEET_BANCI if SHEET_BANCI is not None else 0
    df_banci = pd.read_excel(INPUT_BANCI, sheet_name=banci_sheet_name)

df_banci = clean_col_names(df_banci)
df_banci = standardize_columns(df_banci, BANCI_COLUMN_MAPPING, strict=False, verbose=True)
print("  📌 Preservando copias de columnas originales con sufijo _banci...")
for col in list(df_banci.columns):
    if not col.endswith("_banci"):
        df_banci[f"{col}_banci"] = df_banci[col].copy()

_COLS_BANCI_OPCIONALES = ["fue_delito", "delito_hom", "nom_mun_hchos", "id_ci"]
cols_requeridas = [col for col in COLS_BANCI_TARGET if col not in _COLS_BANCI_OPCIONALES]
validar_columnas(df_banci, cols_requeridas, "BANCI")

# Validación histórica se calcula antes de derivar o filtrar por entidad.
referencia_banci = leer_referencia(ETL_BANCI_OUTPUT_FILE)
resumen_banci = validar_entrega_banci(
    df_banci, INPUT_BANCI, referencia=referencia_banci,
    estricta=VALIDACION_ESTRICTA, verbose=True,
)
print(f"\nEntrada estandarizada: {df_banci.shape[0]:,} filas × {df_banci.shape[1]} columnas")

## 4. Transformaciones BANCI (lógica del ETL original)

Esta celda conserva el orden y las reglas del pipeline original: copia de origen, reparación de nombres, validez de identidad, fechas, identificadores, entidad, clasificación de carpeta y bandera de omisión de matching.

In [ ]:
try:
    print("\n--- Generando variables limpias y derivadas (_derivada_banci) ---")
    df_banci = reparar_nombres_mal_formados(df_banci)

    for c in ["nombres", "primer_apellido", "segundo_apellido"]:
        if c in df_banci.columns:
            df_banci[c] = df_banci[c].astype(str).replace(r"^0$", np.nan, regex=True).replace("nan", np.nan)
            df_banci[f"{c}_derivada_banci"] = df_banci[c].copy()

    col_b_nombres, col_b_primer_apellido, col_b_segundo_apellido = "nombres", "primer_apellido", "segundo_apellido"
    nom_raw = (df_banci[col_b_nombres].fillna("").astype(str) + " "
               + df_banci[col_b_primer_apellido].fillna("").astype(str) + " "
               + df_banci[col_b_segundo_apellido].fillna("").astype(str))
    df_banci["nombre_completo_derivada_banci"] = normalizar_texto(nom_raw)

    patron_ruido = r"SE DESCONOCE|RESTOS OSEOS|CUERPO|SIN VIDA|DESCONOCIDO|NO IDENTIFICADO|\bSD\b|\bNA\b|^M$|([a-z]\.){2,}[a-z]?"
    s_nom = df_banci["nombre_completo_derivada_banci"].fillna("")
    cond_not_na = df_banci["nombre_completo_derivada_banci"].notna()
    cond_no_digit = ~s_nom.str.contains(r"\d", regex=True)
    cond_no_ruido = ~s_nom.str.contains(patron_ruido, flags=re.IGNORECASE, regex=True)
    cond_len = s_nom.str.len() >= 11
    df_banci["es_nombre_valido_derivada_banci"] = cond_not_na & cond_no_digit & cond_no_ruido & cond_len

    if "curp" in df_banci.columns:
        df_banci["curp_valida_derivada_banci"] = es_curp_valida_vectorizado(limpiar_curp(df_banci["curp"]).astype(str))
    else:
        df_banci["curp_valida_derivada_banci"] = False

    fecha_corte = datetime.now().strftime("%Y-%m-%d")
    if "fha_de_hchos" in df_banci.columns:
        df_banci["fha_de_hchos_derivada_banci"] = pd.to_datetime(df_banci["fha_de_hchos"], errors="coerce")
        df_banci = corregir_anio_futuro(df_banci, "fha_de_hchos_derivada_banci", fecha_corte)
    else:
        df_banci["fha_de_hchos_derivada_banci"] = pd.NaT
    if "fha_nac" in df_banci.columns:
        df_banci["fha_nac_derivada_banci"] = pd.to_datetime(df_banci["fha_nac"], errors="coerce")
    else:
        df_banci["fha_nac_derivada_banci"] = pd.NaT

    for col_std, der_col in [("id_mun_hchos", "id_mun_hchos_derivada_banci"), ("llave", "llave_derivada_banci"), ("folio_rnpdno", "fub_derivada_banci")]:
        if col_std in df_banci.columns:
            df_banci[der_col] = df_banci[col_std].astype(str).replace("nan", np.nan).replace("None", np.nan)
        elif f"{col_std}_banci" in df_banci.columns:
            df_banci[der_col] = df_banci[f"{col_std}_banci"].astype(str).replace("nan", np.nan).replace("None", np.nan)
        else:
            df_banci[der_col] = np.nan
        df_banci[col_std] = df_banci[der_col]

    col_estatus = next((k for k, v in BANCI_COLUMN_MAPPING.items() if "estatus" in [x.lower() for x in v] or k == "localizado_o_no_localizado"), "localizado_o_no_localizado")
    if col_estatus in df_banci.columns:
        df_banci["localizado_o_no_localizado_derivada_banci"] = df_banci[col_estatus].astype(str).str.strip().str.upper()
    else:
        df_banci["localizado_o_no_localizado_derivada_banci"] = "NO LOCALIZADA"
    df_banci["entidad_derivada_banci"] = normalizar_texto(df_banci["entidad"]) if "entidad" in df_banci.columns else None

    for _col_ntra in ["NTRA_CI", "ntra_ci", "NtraCi", "ntraci", "ntra_ci_banci"]:
        if _col_ntra in df_banci.columns:
            df_banci["ntra_ci_derivada_banci"] = df_banci[_col_ntra].astype(str).str.strip().replace({"nan":np.nan,"None":np.nan,"<NA>":np.nan,"":np.nan})
            n_nulos = df_banci["ntra_ci_derivada_banci"].isna().sum()
            print(f"  ntra_ci_derivada_banci desde {_col_ntra!r}: {len(df_banci)-n_nulos:,} no nulos")
            break
    else:
        df_banci["ntra_ci_derivada_banci"] = np.nan
        print("  NTRA_CI no encontrada: motor refinado no hará folio-match.")

    for _col_idci in ["ID_CI", "id_ci", "idci", "id_ci_banci"]:
        if _col_idci in df_banci.columns:
            df_banci["id_ci_derivada_banci"] = df_banci[_col_idci].astype(str).str.strip().replace({"nan":np.nan,"None":np.nan,"<NA>":np.nan,"":np.nan})
            n_nulos = df_banci["id_ci_derivada_banci"].isna().sum()
            print(f"  id_ci_derivada_banci desde {_col_idci!r}: {len(df_banci)-n_nulos:,} no nulos")
            break
    else:
        df_banci["id_ci_derivada_banci"] = np.nan
        print("  ID_CI no encontrada: los folios con consecutivo corto no tendrán respaldo.")

    for _col_mun in ["nom_mun_hchos", "NOM_MUN_HCHOS", "municipio_hechos", "municipio"]:
        if _col_mun in df_banci.columns:
            df_banci["nom_mun_hchos_derivada_banci"] = normalizar_texto(df_banci[_col_mun])
            n_nulos = df_banci["nom_mun_hchos_derivada_banci"].isna().sum()
            print(f"  nom_mun_hchos_derivada_banci desde {_col_mun!r}: {len(df_banci)-n_nulos:,} no nulos")
            break
    else:
        df_banci["nom_mun_hchos_derivada_banci"] = np.nan
        print("  NOM_MUN_HCHOS no encontrada: el municipio no se usará como corroborador.")

    _carp = df_banci["ntra_ci_derivada_banci"].fillna("").astype(str).str.strip()
    _idci = df_banci["id_ci_derivada_banci"].fillna("").astype(str).str.strip()
    _carp_real = ~(_carp.eq("") | _carp.str.fullmatch(r"0+", na=False))
    _parte = normalizar_texto(_carp.where(_carp_real, _idci))
    _ent = normalizar_texto(df_banci["entidad_derivada_banci"])
    df_banci["id_carpeta_derivada_banci"] = (_ent + "/" + _parte).where(_parte.str.strip() != "", np.nan)
    _n = int(df_banci["id_carpeta_derivada_banci"].notna().sum())
    print(f"  id_carpeta_derivada_banci: {_n:,} construidos ({int((~_carp_real).sum()):,} desde ID_CI por carpeta vacía o en ceros)")

    col_fub_check = "fub_derivada_banci"
    cond_identidad_invalida = (~df_banci["curp_valida_derivada_banci"]) & (~df_banci["es_nombre_valido_derivada_banci"])
    if col_fub_check in df_banci.columns:
        cond_salvados_por_fub = df_banci[col_fub_check].notna() & (df_banci[col_fub_check] != "") & (df_banci[col_fub_check] != "nan")
        mask_invalido = cond_identidad_invalida & (~cond_salvados_por_fub)
    else:
        mask_invalido = cond_identidad_invalida
    df_banci["skip_match_derivada_banci"] = False
    df_banci.loc[mask_invalido, "skip_match_derivada_banci"] = True
    df_banci["comentario_match_derivada_banci"] = None
    df_banci.loc[mask_invalido, "comentario_match_derivada_banci"] = "Identidad Inválida (Sin Nombre ni CURP válidos)"

    if ESTADO:
        estado_filtro = ENTIDADES.get(ESTADO, "").upper()
        if "entidad_derivada_banci" in df_banci.columns:
            print(f"  🔍 Filtrando BANCI por '{estado_filtro}' en 'entidad_derivada_banci'...")
            ent_vals = df_banci["entidad_derivada_banci"].fillna("")
            estado_filtro_norm = normalizar_texto(estado_filtro)
            mask = ent_vals == estado_filtro_norm
            n_antes = len(df_banci)
            df_banci = df_banci[mask].copy()
            print(f"    🔻 BANCI Filtrado: {n_antes} -> {len(df_banci)} registros.")

    print("\n🔨 Homogeneizando tipos en columnas object para exportación a Parquet...")
    for col_name in df_banci.columns:
        if df_banci[col_name].dtype == "object":
            df_banci[col_name] = df_banci[col_name].astype(str).replace(r"^(nan|None|<NA>|NAN|NONE)$", np.nan, regex=True)
            df_banci[col_name] = df_banci[col_name].where(df_banci[col_name].notna() & (df_banci[col_name] != "nan"), np.nan)
    df_banci_derivado = df_banci.copy()
    df_banci = df_banci[BANCI_OUTPUT_COLUMNS]
except Exception as error:
    raise RuntimeError(f"❌ ERROR CRÍTICO cargando/procesando BANCI: {error}") from error

print(f"DataFrame de salida: {df_banci.shape[0]:,} filas × {df_banci.shape[1]} columnas")
display(df_banci.head())

## 5. Resumen y exportación autónoma

La referencia de validación sigue siendo el Parquet oficial de la última corrida; para no sobrescribirlo, esta ejecución escribe sobre el destino ETL compartido. El formato de archivo y el orden de columnas conservan el contrato BANCI.

In [ ]:
print("Conteo de registros por estatus derivado:")
display(df_banci["localizado_o_no_localizado_derivada_banci"].value_counts(dropna=False).to_frame("registros"))
print("\nIndicadores de matching:")
print(f"  CURP válidas: {int(df_banci_derivado['curp_valida_derivada_banci'].sum()):,}")
print(f"  Nombre válido: {int(df_banci_derivado['es_nombre_valido_derivada_banci'].sum()):,}")
print(f"  Omitidos del matching: {int(df_banci_derivado['skip_match_derivada_banci'].sum()):,}")

if EXPORTAR_RESULTADO and not df_banci.empty:
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    if FORMATO_SALIDA.lower() == "parquet":
        escribir_parquet_con_resumen(df_banci, OUTPUT_FILE, resumen_banci)
        archivo_salida = OUTPUT_FILE
    elif FORMATO_SALIDA.lower() == "csv":
        archivo_salida = OUTPUT_FILE.with_suffix(".csv")
        df_banci.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
    else:
        raise ValueError("FORMATO_SALIDA debe ser 'parquet' o 'csv'.")
    if not archivo_salida.is_file() or archivo_salida.stat().st_size == 0:
        raise IOError(f"No se generó correctamente el archivo: {archivo_salida}")
    print(f"\n✅ Archivo autónomo exportado: {archivo_salida}")
    print(f"Filas: {len(df_banci):,} | columnas: {len(df_banci.columns)} | tamaño: {archivo_salida.stat().st_size:,} bytes")
else:
    print("No se exportó: salida vacía o EXPORTAR_RESULTADO=False.")